In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd
from lake_ice_helpers import open_era5_met_csv


In [ ]:
data_path = None #Add your path here
# Here we assume that era5 data is saved in csv-file:
era5_met = open_era5_met_csv(path_to_file=data_path+r'\era5_met.csv')

In [4]:
era5_met.head()

,station_id,date,tair,tp,sf,day,month,year
1,3_1,1960-01-01,270.935638,0.010248,1.024122e-02,1,1,1960
2,3_1,1960-01-02,268.549255,0.008458,8.453619e-03,2,1,1960
3,3_1,1960-01-03,265.638245,0.000001,5.234033e-07,3,1,1960
4,3_1,1960-01-04,264.218536,0.000015,1.340872e-05,4,1,1960
5,3_1,1960-01-05,265.721497,0.000906,9.025466e-04,5,1,1960


In [5]:
era5_met['tair'] -= 273.15 # Changing temperature to celcius
era5_met['freeze_thaw'] = ((era5_met['tair'] <= 0.5) & (era5_met['tair'] >= -0.5)).astype(int) # Defining freeze-thaw events

In [14]:
np.random.seed(27)
era5_met.iloc[np.random.randint(0,len(era5_met),size=20),:]

,station_id,date,tair,tp,sf,day,month,year,freeze_thaw
16520212,667_1,1976-04-27,0.427240,0.000009,0.000006,27,4,1976,1
18206393,744_1,2013-06-15,9.142389,0.005085,0.000000,15,6,2013,0
8949577,415_1,2017-07-02,17.436884,0.000054,0.000000,2,7,2017,0
3684896,177_1,1986-09-16,4.274561,0.012646,0.003469,16,9,1986,0
4390713,195_1,1984-03-07,-1.332190,0.000324,0.000223,7,3,1984,0
3503398,173_1,2006-01-12,0.039301,0.000811,0.000143,12,1,2006,1
7625753,331_1,2005-11-20,-14.882513,0.000380,0.000377,20,11,2005,0
2440730,120_1,1999-01-11,-5.043555,0.000066,0.000062,11,1,1999,0
20549774,837_1,1979-09-15,2.230890,0.001463,0.000000,15,9,1979,0
12430194,532_1,2001-03-01,-26.095984,0.000077,0.000074,1,3,2001,0


In [15]:
era5_met.describe()

,date,tair,tp,sf,day,month,year,freeze_thaw
count,28246042,2.824604e+07,2.824604e+07,2.824604e+07,2.824604e+07,2.824604e+07,2.824604e+07,2.824604e+07
mean,1992-03-31 11:59:59.999992448,2.282693e+00,2.254637e-03,7.708908e-04,1.572922e+01,6.499576e+00,1.991750e+03,4.493532e-02
min,1960-01-01 00:00:00,-5.765485e+01,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.960000e+03,0.000000e+00
25%,1976-02-15 00:00:00,-4.102515e+00,4.457459e-05,0.000000e+00,8.000000e+00,4.000000e+00,1.976000e+03,0.000000e+00
50%,1992-03-31 12:00:00,2.481729e+00,5.232217e-04,7.636845e-08,1.600000e+01,6.000000e+00,1.992000e+03,0.000000e+00
75%,2008-05-16 00:00:00,1.070873e+01,2.604544e-03,3.542583e-04,2.300000e+01,9.000000e+00,2.008000e+03,0.000000e+00
max,2024-06-30 00:00:00,3.473202e+01,1.376251e-01,1.069524e-01,3.100000e+01,1.200000e+01,2.024000e+03,1.000000e+00
std,NaN,1.076975e+01,4.162142e-03,2.259672e-03,8.799841e+00,3.448809e+00,1.862020e+01,2.071621e-01


In [16]:
era5_met.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28246042 entries, 1 to 28246042
Data columns (total 9 columns):
 #   Column       Dtype         
---  ------       -----         
 0   station_id   object        
 1   date         datetime64[ns]
 2   tair         float64       
 3   tp           float64       
 4   sf           float64       
 5   day          int32         
 6   month        int32         
 7   year         int32         
 8   freeze_thaw  int32         
dtypes: datetime64[ns](1), float64(3), int32(4), object(1)
memory usage: 1.7+ GB


In [17]:
era5_met = era5_met.drop(columns=['date'])

In [18]:
era5_met.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28246042 entries, 1 to 28246042
Data columns (total 8 columns):
 #   Column       Dtype  
---  ------       -----  
 0   station_id   object 
 1   tair         float64
 2   tp           float64
 3   sf           float64
 4   day          int32  
 5   month        int32  
 6   year         int32  
 7   freeze_thaw  int32  
dtypes: float64(3), int32(4), object(1)
memory usage: 1.5+ GB


## Creating summary table with dask parallel computing tools
* Below is just one possible way to do this.
* I dropped away all 'count'-columns during network-training. These were meaningless for modelling.
* Both 'mean' and 'median' are included.


In [19]:
era5_ddf = dd.from_pandas(era5_met,npartitions=6)

In [20]:
drop_cols = ['station_id','date','day','month','year']
era5_summary = era5_ddf.groupby(['station_id','year','month']).agg({col: ['count',
                                                                  'median',
                                                                  'mean',
                                                                  'std',
                                                                  'min',
                                                                  'max',
                                                                  lambda x: x.quantile(0.025),
                                                                  lambda x: x.quantile(0.25),
                                                                  lambda x: x.quantile(0.75),
                                                                  lambda x: x.quantile(0.975)]
                                                    for col in era5_ddf.columns if col not in drop_cols}).compute()

In [21]:
era5_summary.columns

MultiIndex([(       'tair',      'count'),
            (       'tair',     'median'),
            (       'tair',       'mean'),
            (       'tair',        'std'),
            (       'tair',        'min'),
            (       'tair',        'max'),
            (       'tair', '<lambda_0>'),
            (       'tair', '<lambda_1>'),
            (       'tair', '<lambda_2>'),
            (       'tair', '<lambda_3>'),
            (         'tp',      'count'),
            (         'tp',     'median'),
            (         'tp',       'mean'),
            (         'tp',        'std'),
            (         'tp',        'min'),
            (         'tp',        'max'),
            (         'tp', '<lambda_0>'),
            (         'tp', '<lambda_1>'),
            (         'tp', '<lambda_2>'),
            (         'tp', '<lambda_3>'),
            (         'sf',      'count'),
            (         'sf',     'median'),
            (         'sf',       'mean'),
           

In [22]:
era5_summary.rename(columns={'<lambda_0>':'2.5%','<lambda_1>':'25%','<lambda_2>':'75%','<lambda_3>':'97.5%'},
                    inplace=True)

In [23]:
era5_summary

tair                                             \
                      count     median       mean       std        min   
station_id year month                                                    
619_1      1992 12       31   1.404810   0.685883  3.140089  -4.374792   
                2        29   2.032831   1.073390  3.245998  -6.044135   
                3        31   3.579370   3.092313  2.275664  -1.529272   
                7        31  17.598749  18.094283  2.083536  14.946710   
           1993 12       31  -0.147986   0.058927  2.915075  -9.299628   
...                     ...        ...        ...       ...        ...   
           1991 8        31  16.895044  17.210069  2.202977  13.596826   
                9        30  12.262308  12.560668  2.635328   8.514886   
           1992 1        31   0.638239   1.153179  3.259450  -4.911658   
                10       31   3.968774   4.303812  3.157202  -2.492102   
                11       30   2.444818   2.429306  2.165530  -1.855780   

                                                                              \
                             max       2.5%        25%        75%      97.5%   
station_id year month                                                          
619_1      1992 12      6.307977  -4.252798  -1.764960   2.607965   5.874956   
                2       6.060114  -5.643762  -0.964148   2.978998   5.615649   
                3       6.725549  -0.956587   1.287256   4.670541   6.667894   
                7      23.063562  15.176233  16.412515  19.717401  21.778780   
           1993 12      6.352319  -6.131743  -0.582632   0.926370   4.948724   
...                          ...        ...        ...        ...        ...   
           1991 8      21.225854  13.770273  15.511560  18.843118  21.185251   
                9      18.760950   8.661555  11.035173  13.515916  18.120778   
           1992 1       8.584711  -4.506125  -1.096228   3.401758   7.107942   
                10     10.566705  -1.573920   2.278284   7.279581   9.933641   
                11      6.252863  -1.354909   1.497659   4.259622   5.807902   

                       ... freeze_thaw                                     \
                       ...       count median      mean       std min max   
station_id year month  ...                                                  
619_1      1992 12     ...          31    0.0  0.000000  0.000000   0   0   
                2      ...          29    0.0  0.103448  0.309934   0   1   
                3      ...          31    0.0  0.096774  0.300537   0   1   
                7      ...          31    0.0  0.000000  0.000000   0   0   
           1993 12     ...          31    0.0  0.322581  0.475191   0   1   
...                    ...         ...    ...       ...       ...  ..  ..   
           1991 8      ...          31    0.0  0.000000  0.000000   0   0   
                9      ...          30    0.0  0.000000  0.000000   0   0   
           1992 1      ...          31    0.0  0.064516  0.249731   0   1   
                10     ...          31    0.0  0.032258  0.179605   0   1   
                11     ...          30    0.0  0.066667  0.253708   0   1   

                                            
                      2.5%  25%  75% 97.5%  
station_id year month                       
619_1      1992 12     0.0  0.0  0.0  0.00  
                2      0.0  0.0  0.0  1.00  
                3      0.0  0.0  0.0  1.00  
                7      0.0  0.0  0.0  0.00  
           1993 12     0.0  0.0  1.0  1.00  
...                    ...  ...  ...   ...  
           1991 8      0.0  0.0  0.0  0.00  
                9      0.0  0.0  0.0  0.00  
           1992 1      0.0  0.0  0.0  1.00  
                10     0.0  0.0  0.0  0.25  
                11     0.0  0.0  0.0  1.00  

[928026 rows x 40 columns]

Uncomment this to save the data frame into csv:

In [ ]:
#era5_summary.to_csv(data_path+'\\era5_monthly_summary_celsius.csv')

Here you can test if data frame is opened correctly:

In [35]:
era5_summary_file = pd.read_csv(data_path+r'\era5_monthly_summary_celsius.csv',
                                header=[0,1],
                                index_col=[0,1,2],
                                low_memory=False)

In [36]:
era5_summary_file

tair                                             \
                      count     median       mean       std        min   
station_id year month                                                    
619_1      1992 12       31   1.404810   0.685883  3.140089  -4.374792   
                2        29   2.032831   1.073390  3.245998  -6.044135   
                3        31   3.579370   3.092313  2.275664  -1.529272   
                7        31  17.598749  18.094283  2.083536  14.946710   
           1993 12       31  -0.147986   0.058927  2.915075  -9.299628   
...                     ...        ...        ...       ...        ...   
           1991 8        31  16.895044  17.210069  2.202977  13.596826   
                9        30  12.262308  12.560668  2.635328   8.514886   
           1992 1        31   0.638239   1.153179  3.259450  -4.911658   
                10       31   3.968774   4.303812  3.157202  -2.492102   
                11       30   2.444818   2.429306  2.165530  -1.855780   

                                                                              \
                             max       2.5%        25%        75%      97.5%   
station_id year month                                                          
619_1      1992 12      6.307977  -4.252798  -1.764960   2.607965   5.874956   
                2       6.060114  -5.643762  -0.964148   2.978998   5.615649   
                3       6.725549  -0.956587   1.287256   4.670541   6.667894   
                7      23.063562  15.176233  16.412515  19.717401  21.778780   
           1993 12      6.352319  -6.131743  -0.582632   0.926370   4.948724   
...                          ...        ...        ...        ...        ...   
           1991 8      21.225854  13.770273  15.511560  18.843118  21.185251   
                9      18.760950   8.661555  11.035173  13.515916  18.120778   
           1992 1       8.584711  -4.506125  -1.096228   3.401758   7.107942   
                10     10.566705  -1.573920   2.278284   7.279581   9.933641   
                11      6.252863  -1.354909   1.497659   4.259622   5.807902   

                       ... freeze_thaw                                     \
                       ...       count median      mean       std min max   
station_id year month  ...                                                  
619_1      1992 12     ...          31    0.0  0.000000  0.000000   0   0   
                2      ...          29    0.0  0.103448  0.309934   0   1   
                3      ...          31    0.0  0.096774  0.300537   0   1   
                7      ...          31    0.0  0.000000  0.000000   0   0   
           1993 12     ...          31    0.0  0.322581  0.475191   0   1   
...                    ...         ...    ...       ...       ...  ..  ..   
           1991 8      ...          31    0.0  0.000000  0.000000   0   0   
                9      ...          30    0.0  0.000000  0.000000   0   0   
           1992 1      ...          31    0.0  0.064516  0.249731   0   1   
                10     ...          31    0.0  0.032258  0.179605   0   1   
                11     ...          30    0.0  0.066667  0.253708   0   1   

                                            
                      2.5%  25%  75% 97.5%  
station_id year month                       
619_1      1992 12     0.0  0.0  0.0  0.00  
                2      0.0  0.0  0.0  1.00  
                3      0.0  0.0  0.0  1.00  
                7      0.0  0.0  0.0  0.00  
           1993 12     0.0  0.0  1.0  1.00  
...                    ...  ...  ...   ...  
           1991 8      0.0  0.0  0.0  0.00  
                9      0.0  0.0  0.0  0.00  
           1992 1      0.0  0.0  0.0  1.00  
                10     0.0  0.0  0.0  0.25  
                11     0.0  0.0  0.0  1.00  

[928026 rows x 40 columns]

In [37]:
era5_summary_file.shape

(928026, 40)

In [38]:
era5_summary_file.columns = era5_summary_file.columns.map('_'.join)

In [39]:
era5_summary_file.unstack()

tair_count                                                  \
month                   1     2     3     4     5     6     7     8     9    
station_id year                                                              
1000_1     1960       31.0  29.0  31.0  30.0  31.0  30.0  31.0  31.0  30.0   
           1961       31.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  30.0   
           1962       31.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  30.0   
           1963       31.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  30.0   
           1964       31.0  29.0  31.0  30.0  31.0  30.0  31.0  31.0  30.0   
...                    ...   ...   ...   ...   ...   ...   ...   ...   ...   
9_1        2020       31.0  29.0  31.0  30.0  31.0  30.0  31.0  31.0  30.0   
           2021       31.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  30.0   
           2022       31.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  30.0   
           2023       31.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  30.0   
           2024       31.0  29.0  31.0  30.0  31.0  30.0   NaN   NaN   NaN   

                       ... freeze_thaw_97.5%                                   \
month              10  ...                3      4     5    6    7    8    9    
station_id year        ...                                                      
1000_1     1960  31.0  ...               0.0  1.000  0.00  0.0  0.0  0.0  0.0   
           1961  31.0  ...               1.0  1.000  0.25  0.0  0.0  0.0  0.0   
           1962  31.0  ...               1.0  0.000  0.00  0.0  0.0  0.0  0.0   
           1963  31.0  ...               1.0  1.000  0.00  0.0  0.0  0.0  0.0   
           1964  31.0  ...               0.0  1.000  0.00  0.0  0.0  0.0  0.0   
...               ...  ...               ...    ...   ...  ...  ...  ...  ...   
9_1        2020  31.0  ...               1.0  0.000  0.00  0.0  0.0  0.0  0.0   
           2021  31.0  ...               1.0  0.000  0.00  0.0  0.0  0.0  0.0   
           2022  31.0  ...               1.0  0.275  0.00  0.0  0.0  0.0  0.0   
           2023  31.0  ...               1.0  1.000  0.00  0.0  0.0  0.0  0.0   
           2024   NaN  ...               1.0  0.000  0.00  0.0  NaN  NaN  NaN   

                                   
month             10     11    12  
station_id year                    
1000_1     1960  1.0  1.000  0.25  
           1961  0.0  1.000  0.25  
           1962  1.0  1.000  0.00  
           1963  0.0  1.000  0.25  
           1964  0.0  0.275  0.00  
...              ...    ...   ...  
9_1        2020  0.0  0.275  1.00  
           2021  0.0  1.000  0.00  
           2022  0.0  1.000  1.00  
           2023  0.0  1.000  1.00  
           2024  NaN    NaN   NaN  

[77935 rows x 480 columns]